# Restaurant Revenue Prediction — 01 Exploratory Data Analysis

**Regression** project on the TFI (Kaggle) restaurant dataset. Goal: predict a restaurant's annual `revenue` from its open date, city, city group, type, and 37 anonymised numeric features `P1`–`P37` (obfuscated for privacy).

This is a **small, hard dataset — only 137 rows** — so EDA is about finding which few signals actually relate to revenue, and setting realistic expectations for modelling.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline
import utils

df = utils.load_data()
print('shape:', df.shape)
df.head()

## 1. Target: revenue distribution

Revenue is right-skewed with a long tail (a few very high earners), which will hurt linear models unless regularised.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(df['revenue'], bins=30, color='#4c72b0'); ax[0].set_title('revenue'); ax[0].set_xlabel('USD')
ax[1].hist(np.log1p(df['revenue']), bins=30, color='#55a868'); ax[1].set_title('log(1+revenue)')
plt.tight_layout(); plt.show()
print(df['revenue'].describe().round(0).to_string())

## 2. Revenue by City Group and Type

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
sns.barplot(x='City Group', y='revenue', data=df, ax=ax[0], estimator=np.mean, errorbar=None)
ax[0].set_title('Mean revenue by City Group')
sns.barplot(x='Type', y='revenue', data=df, ax=ax[1], estimator=np.mean, errorbar=None)
ax[1].set_title('Mean revenue by Type')
plt.tight_layout(); plt.show()
print(df.groupby('City Group')['revenue'].agg(['mean','count']).round(0).to_string())
print()
print(df.groupby('Type')['revenue'].agg(['mean','count']).round(0).to_string())

## 3. Restaurant age (from Open Date)

`Open Date` itself is not a model feature, but the **age** of the restaurant (years since opening, measured from a 2015-01-01 snapshot) might be. Older restaurants tend to be established earners.

In [ ]:
fe = utils.create_features(df)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(fe['restaurant_age_years'], bins=25, color='#c44e52'); ax[0].set_title('restaurant age (years)')
ax[1].scatter(fe['restaurant_age_years'], fe['revenue'], alpha=0.5)
ax[1].set_xlabel('age (years)'); ax[1].set_ylabel('revenue'); ax[1].set_title('age vs revenue')
plt.tight_layout(); plt.show()
print('age stats:'); print(fe['restaurant_age_years'].describe().round(2).to_string())
print('\nage-revenue correlation:', round(fe['restaurant_age_years'].corr(fe['revenue']), 4))

## 4. The P1–P37 obfuscated features

These are anonymised. Many contain a large share of zeros (placeholder/missing). We rank them by absolute correlation with revenue — none is individually strong.

In [ ]:
p_cols = [c for c in df.columns if c.startswith('P')]
pcorr = df[p_cols].corrwith(df['revenue']).abs().sort_values(ascending=False)
print('zero-fraction (max across P-cols):', round((df[p_cols] == 0).mean().max(), 2))
print('\nTop 6 P-features by |correlation| with revenue:')
print(pcorr.head(6).round(3).to_string())
fig, ax = plt.subplots(figsize=(9, 3))
pcorr.head(12).plot(kind='bar', ax=ax, color='#8172b3'); ax.set_title('|corr| with revenue (top 12 P-features)')
plt.tight_layout(); plt.show()

## 5. Correlation heatmap (strongest signals)

In [ ]:
top_p = pcorr.head(5).index.tolist()
sub = fe[['restaurant_age_years'] + top_p + ['revenue']]
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(sub.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation: age + top P-features + revenue'); plt.tight_layout(); plt.show()

## 6. Summary of findings

- **137 rows, 0 missing values, 43 columns.** This is a very small dataset for 40+ candidate features — overfitting is the central risk.
- **revenue is right-skewed** (mean ≈ \$4.45M, max ≈ \$19.7M).
- **City Group matters**: Big-City restaurants average ≈ \$4.98M vs ≈ \$3.75M for Other. **Type** is less clear and `DT` has only 1 row.
- **Restaurant age is the single strongest signal** (corr ≈ **0.33** with revenue) — older restaurants earn more.
- **The P-features are individually weak** (top |corr| = P2 at 0.19) and many are zero-filled. Predictive power, if any, is diffuse.
- Expect **modest model performance**; notebooks 02–03 engineer features and quantify exactly how predictable revenue is.